In [8]:
!pip install rasterio geopandas kornia

In [9]:
import gc
import json
import os
import time
from collections import defaultdict
from datetime import datetime

import cv2
import numpy as np
import psutil
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import Polygon, box as shapely_box
from shapely.ops import unary_union
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

In [10]:
!nvidia-smi

Tue Apr 14 02:55:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [50]:
# ============================================================================
# FRACTAL RESUNET ARCHITECTURE
# ============================================================================

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, max(channels//reduction, 4)), nn.ReLU(),
            nn.Linear(max(channels//reduction, 4), channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 1)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(x))

class FracTALUnit(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.ch_att = ChannelAttention(channels)
        self.sp_att = SpatialAttention(channels)
        self.alpha  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        a = torch.clamp(self.alpha, 0.0, 1.0)
        return a * self.ch_att(x) + (1-a) * self.sp_att(x)

class FracTALResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
        )
        self.fractal = FracTALUnit(out_ch)
        self.proj    = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.proj(x) + self.fractal(self.conv(x))

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks=2, dilation=1):
        super().__init__()
        blocks = [FracTALResBlock(in_ch, out_ch, dilation)]
        for _ in range(n_blocks-1):
            blocks.append(FracTALResBlock(out_ch, out_ch, dilation))
        self.blocks = nn.Sequential(*blocks)
        self.pool   = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.blocks(x)
        return self.pool(skip), skip

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.block = FracTALResBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))

class FracTALResUNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=2, base_ch=32):
        super().__init__()
        c = base_ch
        self.enc1 = EncoderBlock(in_ch, c,    n_blocks=2)
        self.enc2 = EncoderBlock(c,     c*2,  n_blocks=2)
        self.enc3 = EncoderBlock(c*2,   c*4,  n_blocks=3)
        self.enc4 = EncoderBlock(c*4,   c*8,  n_blocks=3)
        self.bottleneck = nn.Sequential(
            FracTALResBlock(c*8,  c*16, dilation=2),
            FracTALResBlock(c*16, c*16, dilation=4),
            FracTALResBlock(c*16, c*16, dilation=2),
        )
        self.dec4 = DecoderBlock(c*16, c*8,  c*8)
        self.dec3 = DecoderBlock(c*8,  c*4,  c*4)
        self.dec2 = DecoderBlock(c*4,  c*2,  c*2)
        self.dec1 = DecoderBlock(c*2,  c,    c)
        self.seg_head   = nn.Conv2d(c, num_classes, 1)
        self.bound_head = nn.Conv2d(c, 1, 1)
        self.dist_head  = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return {'seg': self.seg_head(x), 'bound': self.bound_head(x), 'dist': self.dist_head(x)}


# ============================================================================
# GPU SETUP
# ============================================================================

MEM_LIMIT_GB = 8.0

def _gpu_available():
    return torch.cuda.is_available()

def setup_gpu_memory(vram_limit_gb=12.0):
    if not _gpu_available():
        print('  No GPU detected — running on CPU')
        return
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    frac     = min(vram_limit_gb / total_gb, 0.90)
    torch.cuda.set_per_process_memory_fraction(frac)
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {total_gb:.1f} GB total  →  {frac*100:.0f}% reserved ({frac*total_gb:.1f} GB)')


# ============================================================================
# IMAGE READER
# ============================================================================

class ImageReader:
    def __init__(self, image_path: str, mem_limit_gb: float = MEM_LIMIT_GB):
        if not os.path.exists(image_path):
            raise FileNotFoundError(image_path)
        self.image_path    = image_path
        self._cache        = None
        self._cache_done   = False
        self.original_dims = None
        self.transform     = None
        self.crs           = None
        self._load_metadata()
        self._maybe_cache(mem_limit_gb)

    def _load_metadata(self):
        try:
            with rasterio.open(self.image_path) as src:
                self.original_dims = (src.width, src.height)
                self.transform     = src.transform
                self.crs           = src.crs
                self.band_count    = src.count
        except Exception:
            from PIL import Image
            with Image.open(self.image_path) as img:
                w, h = img.size
            self.original_dims = (w, h)
            self.transform     = from_bounds(0, 0, w, h, w, h)
            self.crs           = None
            self.band_count    = 3

    def _maybe_cache(self, mem_limit_gb):
        if self._cache_done: return
        self._cache_done = True
        w, h     = self.original_dims
        image_gb = w * h * 3 / 1024**3
        if image_gb > mem_limit_gb:
            print(f'  Image {image_gb:.1f} GB > limit — windowed reads')
            return
        print(f'  Caching {image_gb:.1f} GB image into RAM…')
        try:
            with rasterio.open(self.image_path) as src:
                if src.count >= 3:
                    data = np.transpose(src.read([1,2,3]), (1,2,0))
                else:
                    data = cv2.cvtColor(src.read(1), cv2.COLOR_GRAY2RGB)
            self._cache = np.clip(data, 0, 255).astype(np.uint8)
            print(f'  ✓ Cached ({self._cache.nbytes/1024**3:.2f} GB)')
        except MemoryError:
            print('  MemoryError — windowed reads only')
            self._cache = None

    def read_crop(self, x1, y1, x2, y2) -> np.ndarray:
        w, h = self.original_dims
        x1 = max(0,min(x1,w));  y1 = max(0,min(y1,h))
        x2 = max(x1,min(x2,w)); y2 = max(y1,min(y2,h))
        if x2<=x1 or y2<=y1: return np.zeros((0,0,3), dtype=np.uint8)
        if self._cache is not None: return self._cache[y1:y2,x1:x2].copy()
        window = Window(x1, y1, x2-x1, y2-y1)
        try:
            with rasterio.open(self.image_path) as src:
                if src.count >= 3:
                    data = np.transpose(src.read([1,2,3], window=window), (1,2,0))
                else:
                    data = cv2.cvtColor(src.read(1, window=window), cv2.COLOR_GRAY2RGB)
            return np.clip(data, 0, 255).astype(np.uint8)
        except Exception as e:
            print(f'  ⚠ read_crop error: {e}')
            return np.zeros((y2-y1, x2-x1, 3), dtype=np.uint8)


# ============================================================================
# FRACTAL RESUNET TILED INFERENCE
#
#   - Runs the FracTAL model on each tile to get a semantic seg mask
#     AND a boundary probability map simultaneously
#   - The boundary map is used to cut the seg mask before contouring
#   - Detections are field *polygons* extracted from the cut seg mask
#   - Accumulate seg logits + boundary probabilities into full-image
#     arrays using a blending weight map.  Polygons are extracted ONCE from
#     the complete mosaicked mask so no field is ever cut at a tile boundary.
# ============================================================================

PREPROCESS = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])


def _tile_to_tensor(tile_np: np.ndarray, tile_size: int) -> torch.Tensor:
    resized = cv2.resize(tile_np, (tile_size, tile_size),
                         interpolation=cv2.INTER_LINEAR)
    return PREPROCESS(Image.fromarray(resized))


# ── Cosine blend window (smooth fade at tile edges to avoid seams) ──────────
def _make_blend_window(tile_size: int) -> np.ndarray:
    """2-D cosine window: weight=1 at centre, tapers smoothly to ~0 at edges."""
    t = np.hanning(tile_size).astype(np.float32)
    w = np.outer(t, t)
    w = w / w.max()          # normalise peak to 1.0
    return w


def _mask_to_polygons_global(seg_mask: np.ndarray,
                               bound_map: np.ndarray,
                               config: dict) -> list:
    """
    Extract field polygons from the full-image mosaicked masks.
    No tile offsets or scale factors needed — coordinates are already
    in original-image pixel space.
    """

    bound_thresh = config.get('bound_thresh', 0.35)
    min_area_px  = config.get('min_field_px', 300)
    smooth_eps   = config.get('smooth_epsilon', 1.5)

    field_bin = (seg_mask > 0).astype(np.uint8)
    hard_bnd  = (bound_map > bound_thresh).astype(np.uint8)
    field_cut = cv2.bitwise_and(field_bin, 1 - hard_bnd)

    # Morphological close: heals tiny gaps left by the boundary mask without
    # merging genuinely separate fields
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)) # 3x3
    field_cut = cv2.morphologyEx(field_cut, cv2.MORPH_CLOSE, kernel, iterations=1)

    n_labels, label_map = cv2.connectedComponents(field_cut, connectivity=8)
    print(f'    Connected components: {n_labels - 1} raw instances')

    detections = []
    for lbl in range(1, n_labels):
        instance = (label_map == lbl).astype(np.uint8)
        if instance.sum() < min_area_px:
            continue

        contours, _ = cv2.findContours(instance, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        cnt    = max(contours, key=cv2.contourArea)
        approx = cv2.approxPolyDP(cnt, smooth_eps, closed=True)
        if len(approx) < 3:
            continue

        poly_pts = approx.reshape(-1, 2).astype(np.int32)
        xs, ys   = poly_pts[:, 0], poly_pts[:, 1]
        bbox     = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]
        conf     = float(bound_map[instance == 1].mean())

        detections.append({
            'polygon_px': poly_pts,
            'bbox_px':    bbox,
            'confidence': conf,
            'class_id':   1,
            'class_name': 'field',
        })

    return detections


def run_fractal_tiled(reader: ImageReader, model, config: dict) -> list:
    """
    Run FracTAL ResUNet on overlapping tiles and mosaic predictions
    before extracting polygons.
    """

    tile_size  = config['tile_size']
    overlap    = config['overlap']
    resolution = config['resolution']
    batch_size = config.get('batch_size', 8)
    device     = torch.device('cuda' if _gpu_available() else 'cpu')

    model.eval()
    model.to(device)

    orig_w, orig_h = reader.original_dims
    sw   = int(orig_w * resolution)   # scaled canvas width
    sh   = int(orig_h * resolution)   # scaled canvas height
    step = tile_size - overlap

    tile_positions = [
        (x, y, min(x + tile_size, sw), min(y + tile_size, sh))
        for y in range(0, sh, step)
        for x in range(0, sw, step)
    ]
    print(f'  {len(tile_positions)} tiles  '
          f'(tile={tile_size}px  overlap={overlap}px  resolution={resolution})')

    # ── Allocate mosaicking buffers (lazy-init seg_acc after first batch) ─────
    seg_acc    = None                                   # (sh, sw, C)
    bound_acc  = np.zeros((sh, sw), dtype=np.float32)
    weight_acc = np.zeros((sh, sw), dtype=np.float32)
    blend_win  = _make_blend_window(tile_size)          # (tile_size, tile_size)

    for batch_start in tqdm(range(0, len(tile_positions), batch_size),
                            desc='FracTAL mosaic'):
        batch   = tile_positions[batch_start: batch_start + batch_size]
        tensors = []
        meta    = []   # (x, y, x_end, y_end, tile_h, tile_w)

        for x, y, x_end, y_end in batch:
            ox  = int(x     / resolution); oy  = int(y     / resolution)
            ox2 = int(x_end / resolution); oy2 = int(y_end / resolution)
            ox  = min(ox,  orig_w - 1);    oy  = min(oy,  orig_h - 1)
            ox2 = min(ox2, orig_w);        oy2 = min(oy2, orig_h)

            tile = reader.read_crop(ox, oy, ox2, oy2)
            if tile.size == 0:
                continue
            tile_h, tile_w = tile.shape[:2]
            tensors.append(_tile_to_tensor(tile, tile_size))
            meta.append((x, y, x_end, y_end, tile_h, tile_w))

        if not tensors:
            continue

        batch_tensor = torch.stack(tensors).to(device, dtype=torch.float16)
        with torch.no_grad():
            out = model(batch_tensor)

        # seg logits (B, C, tile_size, tile_size)
        seg_logits = out['seg'].float().cpu().numpy()
        bound_prob = torch.sigmoid(out['bound']).squeeze(1).float().cpu().numpy()

        num_classes = seg_logits.shape[1]
        if seg_acc is None:
            seg_acc = np.zeros((sh, sw, num_classes), dtype=np.float32)

        for i, (x, y, x_end, y_end, tile_h, tile_w) in enumerate(meta):
            tw = x_end - x   # actual width in scaled-canvas space
            th = y_end - y   # actual height in scaled-canvas space

            seg_tile = seg_logits[i]   # (C, tile_size, tile_size)
            bnd_tile = bound_prob[i]   # (tile_size, tile_size)

            # Resize model output from square tile_size -> actual tile dims
            if (tile_size, tile_size) != (th, tw):
                seg_tile_r = np.stack([
                    cv2.resize(seg_tile[c], (tw, th),
                               interpolation=cv2.INTER_LINEAR)
                    for c in range(num_classes)
                ], axis=0)
                bnd_tile_r = cv2.resize(bnd_tile, (tw, th),
                                        interpolation=cv2.INTER_LINEAR)
                w2d = cv2.resize(blend_win, (tw, th),
                                 interpolation=cv2.INTER_LINEAR)
            else:
                seg_tile_r = seg_tile
                bnd_tile_r = bnd_tile
                w2d        = blend_win

            # Weighted accumulation into full-image buffers
            y1, y2 = y, y + th
            x1, x2 = x, x + tw
            seg_acc[y1:y2, x1:x2, :]  += seg_tile_r.transpose(1, 2, 0) * w2d[..., np.newaxis]
            bound_acc[y1:y2, x1:x2]   += bnd_tile_r * w2d
            weight_acc[y1:y2, x1:x2]  += w2d

        del batch_tensor, out
        if _gpu_available() and batch_start % (batch_size * 10) == 0:
            torch.cuda.empty_cache()

    # ── Divide by accumulated weights to get blended predictions ─────────────
    eps          = 1e-6
    w_safe       = np.maximum(weight_acc, eps)
    seg_blended  = seg_acc   / w_safe[..., np.newaxis]   # (sh, sw, C)
    bnd_blended  = bound_acc / w_safe                    # (sh, sw)

    seg_mask_full  = seg_blended.argmax(axis=2).astype(np.uint8)
    bound_map_full = bnd_blended.astype(np.float32)

    # Resize back to original dims
    if resolution != 1.0:
        seg_mask_full  = cv2.resize(seg_mask_full,  (orig_w, orig_h),
                                    interpolation=cv2.INTER_NEAREST)
        bound_map_full = cv2.resize(bound_map_full, (orig_w, orig_h),
                                    interpolation=cv2.INTER_LINEAR)

    print('  ✓ Full-image mosaic complete — extracting polygons from global mask…')

    # ── Extract polygons ONCE from the complete mosaicked mask ───────────────
    detections = _mask_to_polygons_global(seg_mask_full, bound_map_full, config)
    print(f'  ✓ {len(detections)} raw detections (mosaic-first — no tile-boundary cuts)')
    return detections

def polygon_nms(detections: list[dict], iou_threshold: float = 0.4) -> list[dict]:
    if not detections: return detections
    det = sorted(detections, key=lambda d: d['confidence'], reverse=True)
    shapes = []
    for d in det:
        try:
            p = Polygon(d['polygon_px'])
            shapes.append(p.buffer(0) if not p.is_valid else p)
        except Exception:
            shapes.append(None)
    keep    = []
    removed = set()
    for i in range(len(det)):
        if i in removed or shapes[i] is None: continue
        keep.append(det[i])
        pi = shapes[i]; ai = pi.area
        for j in range(i+1, len(det)):
            if j in removed or shapes[j] is None: continue
            try:
                inter = pi.intersection(shapes[j]).area
                if inter == 0: continue
                union = ai + shapes[j].area - inter
                if union > 0 and inter/union > iou_threshold:
                    removed.add(j)
            except Exception: continue
    print(f'  Polygon NMS: {len(det)} → {len(keep)}  (IoU>{iou_threshold})')
    return keep


# ============================================================================
# GEOREFERENCING
# ============================================================================

def detections_to_geodataframe(detections, transform, crs):
    geo_polygons = []; attrs = []
    for d in tqdm(detections, desc='Georeferencing'):
        try:
            geo_coords = [
                rasterio.transform.xy(transform, float(py), float(px))
                for px, py in d['polygon_px']
            ]
            geo_poly = Polygon(geo_coords)
            if not geo_poly.is_valid: geo_poly = geo_poly.buffer(0)
            if geo_poly.is_empty: continue
        except Exception: continue
        geo_polygons.append(geo_poly)
        attrs.append({
            'confidence':   d['confidence'],
            'class_id':     d['class_id'],
            'class_name':   d['class_name'],
            'area_m2':      geo_poly.area,
            'area_px':      Polygon(d['polygon_px']).area,
            'num_vertices': len(d['polygon_px']),
        })
    if not geo_polygons: return gpd.GeoDataFrame()
    gdf = gpd.GeoDataFrame(attrs, geometry=geo_polygons, crs=crs)
    print(f'  ✓ {len(gdf)} georeferenced polygons')
    return gdf


# ============================================================================
# VISUALISATION
# ============================================================================

def visualize_segmentation(reader, detections, config):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.collections import PatchCollection
    from matplotlib.patches import Polygon as MplPolygon
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    output_dir = config['output_dir']
    name       = config.get('image_name', 'output')
    max_dim    = config.get('max_viz_dimension', 4000)
    dpi        = config.get('viz_dpi', 150)

    orig_w, orig_h = reader.original_dims
    scale  = min(max_dim/orig_w, max_dim/orig_h, 1.0)
    out_w  = int(orig_w * scale)
    out_h  = int(orig_h * scale)
    print(f'  Canvas: {orig_w}×{orig_h} → {out_w}×{out_h}  scale={scale:.4f}')

    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f'{name}_segmentation.png')

    strip_src_h = config.get('viz_strip_height', 512)
    bg = np.zeros((out_h, out_w, 3), dtype=np.uint8)
    for y0 in range(0, out_h, max(1, int(strip_src_h*scale))):
        y1 = min(y0 + max(1, int(strip_src_h*scale)), out_h)
        src = reader.read_crop(0, int(y0/scale), orig_w, int(y1/scale))
        if src.size == 0: continue
        bg[y0:y1] = cv2.resize(src, (out_w, y1-y0), interpolation=cv2.INTER_AREA)

    fig_w = out_w / dpi
    fig_h = out_h / dpi
    fig, ax = plt.subplots(figsize=(fig_w+1.4, fig_h))
    ax.imshow(bg/255.0, extent=[0,out_w,out_h,0], aspect='equal', alpha=0.85)
    ax.set_xlim(0,out_w); ax.set_ylim(out_h,0); ax.set_axis_off()
    ax.set_title('Crop Field Detections — FracTAL ResUNet', fontsize=14, fontweight='bold', pad=10)

    cmap  = cm.plasma
    norm  = mcolors.Normalize(vmin=0.0, vmax=1.0)
    patch_list  = []
    conf_values = []
    for d in detections:
        pts = (d['polygon_px'] * scale).astype(np.float32)
        pts[:,0] = np.clip(pts[:,0], 0, out_w-1)
        pts[:,1] = np.clip(pts[:,1], 0, out_h-1)
        if len(pts) < 3: continue
        patch_list.append(MplPolygon(pts, closed=True))
        conf_values.append(d['confidence'])

    if patch_list:
        face_colors = [cmap(norm(c)) for c in conf_values]
        fill_colors = [(r,g,b,0.25) for r,g,b,_ in face_colors]
        pc = PatchCollection(
            patch_list,
            facecolors=fill_colors,
            edgecolors=[(1.0,0.0,0.0,0.9)]*len(patch_list),
            linewidths=1.2,
        )
        ax.add_collection(pc)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('Boundary Confidence', fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    n     = len(detections)
    avg_c = np.mean(conf_values) if conf_values else 0.0
    ax.text(0.02, 0.98,
            f'Total: {n}\nAvg Boundary Conf: {avg_c:.3f}',
            transform=ax.transAxes, fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      alpha=0.9, edgecolor='black', linewidth=0.8),
            fontfamily='monospace')

    plt.tight_layout(pad=0.5)
    plt.savefig(out_path, dpi=dpi, bbox_inches='tight', pil_kwargs={'compress_level':6})
    plt.close(fig)
    del bg, patch_list
    gc.collect()
    print(f'  ✓ PNG: {out_path}  ({os.path.getsize(out_path)/1024**2:.1f} MB)')
    return out_path


# ============================================================================
# SAVE OUTPUTS
# ============================================================================

def save_outputs(gdf, detections, config, elapsed_sec):
    output_dir = config['output_dir']
    name       = config.get('image_name', 'output')
    os.makedirs(output_dir, exist_ok=True)

    if gdf is not None and len(gdf) > 0 and gdf.crs is not None:
        p = os.path.join(output_dir, f'{name}_fields.geojson')
        try: gdf.to_file(p, driver='GeoJSON'); print(f'  ✓ GeoJSON : {p}')
        except Exception as e: print(f'  ⚠ GeoJSON failed: {e}')

    if gdf is not None and len(gdf) > 0:
        p = os.path.join(output_dir, f'{name}_fields.csv')
        try: gdf.drop(columns='geometry').to_csv(p, index=False); print(f'  ✓ CSV     : {p}')
        except Exception as e: print(f'  ⚠ CSV failed: {e}')

    if detections:
        confs = [d['confidence'] for d in detections]
        areas = [Polygon(d['polygon_px']).area for d in detections if len(d['polygon_px'])>=3]
        stats = {
            'model':                 'FracTAL ResUNet',
            'total_fields':          len(detections),
            'avg_boundary_conf':     float(np.mean(confs)),
            'min_boundary_conf':     float(np.min(confs)),
            'max_boundary_conf':     float(np.max(confs)),
            'avg_area_px':           float(np.mean(areas)) if areas else 0,
            'median_area_px':        float(np.median(areas)) if areas else 0,
            'processing_time_min':   round(elapsed_sec/60, 2),
        }
        p = os.path.join(output_dir, f'{name}_statistics.json')
        try:
            with open(p,'w') as f: json.dump(stats, f, indent=2)
            print(f'  ✓ Stats   : {p}')
        except Exception as e: print(f'  ⚠ Stats failed: {e}')


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def run_cropfield_segmentation(image_path: str, model, config: dict):
    """
    FracTAL ResUNet pipeline:
      1. Load image metadata + optional RAM cache
      2. Tiled FracTAL inference (seg + boundary heads simultaneously)
      3. Cross-tile polygon NMS
      4. Georeferencing → GeoDataFrame
      5. Visualization
      6. Save GeoJSON, CSV, statistics
    """

    t0      = time.time()
    process = psutil.Process()
    _ram    = lambda: process.memory_info().rss / 1024**3

    print(f"\n{'='*65}")
    print('  CROP FIELD SEGMENTATION — FracTAL ResUNet')
    print(f"{'='*65}")
    print(f"  Image      : {image_path}")
    print(f"  Checkpoint : {config.get('model_path', 'provided externally')}")
    print(f"  Output dir : {config['output_dir']}")

    print(f"\n[1/5] Loading image…")
    reader = ImageReader(image_path)
    orig_w, orig_h = reader.original_dims
    print(f"  ✓ {orig_w}×{orig_h}  CRS={reader.crs}  RAM={_ram():.2f} GB")

    print(f"\n[2/5] Running FracTAL ResUNet tiled inference…")
    detections = run_fractal_tiled(reader, model, config)
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    if not detections:
        print('  ⚠ No fields detected. Check bound_thresh and model path.')
        return None, []

    print(f"\n[3/5] Polygon NMS…")
    detections = polygon_nms(detections, config.get('polygon_nms_iou', 0.4))
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    print(f"\n[4/5] Georeferencing…")
    gdf = detections_to_geodataframe(detections, reader.transform, reader.crs)
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    print(f"\n[5/5] Visualization…")
    visualize_segmentation(reader, detections, config)

    elapsed = time.time() - t0
    print(f"\n[Save] Writing outputs…")
    save_outputs(gdf, detections, config, elapsed)

    print(f"\n{'='*65}")
    print(f"  ✓ Done in {elapsed/60:.1f} min  |  {len(detections)} fields")
    print(f"  Peak RAM: {_ram():.2f} GB")
    print(f"{'='*65}\n")
    return gdf, detections

In [53]:
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

setup_gpu_memory(vram_limit_gb=12.0)

# ── Load checkpoint ───────────────────────────────────────────────────────────
MODEL_PATH = (
    '/content/drive/MyDrive/AI-CropFieldSegmentation/result/fractal-resunet-512-v03/best.pt'
)

state_dict  = torch.load(MODEL_PATH, map_location='cpu')
NUM_CLASSES = state_dict['seg_head.weight'].shape[0]
BASE_CH     = state_dict['seg_head.weight'].shape[1]   # infer base_ch too
print(f'Checkpoint → NUM_CLASSES={NUM_CLASSES}  base_ch={BASE_CH}')

model = FracTALResUNet(in_ch=3, num_classes=NUM_CLASSES, base_ch=BASE_CH)
model.load_state_dict(state_dict)
del state_dict   # free CPU RAM

model.eval()
if _gpu_available():
    model = model.cuda()
    model = model.half()   # FP16
    print('  Model loaded in FP16 on GPU')
else:
    print('  Model loaded on CPU')

# ── Config ────────────────────────────────────────────────────────────────────
config = {
    'tile_size':         512,
    'overlap':           128,
    'resolution':        1.0,
    'batch_size':        8,

    'bound_thresh':      0.35,
    'min_field_px':      300,
    'smooth_epsilon':    1.5,

    'polygon_nms_iou':   0.4,

    'max_viz_dimension': 4000,
    'viz_strip_height':  512,

    'output_dir':  '/content/drive/MyDrive/AI-CropFieldSegmentation/output',
    'image_name':  'cropfield_fractal',
    'model_path':  MODEL_PATH,
}

IMAGE_PATH = '/content/drive/MyDrive/AI-CropFieldSegmentation/Screenshot 2026-03-17 063235.png'

gdf, detections = run_cropfield_segmentation(IMAGE_PATH, model, config)

  GPU  : Tesla T4
  VRAM : 14.6 GB total  →  82% reserved (12.0 GB)
Checkpoint → NUM_CLASSES=2  base_ch=32
  Model loaded in FP16 on GPU

  CROP FIELD SEGMENTATION — FracTAL ResUNet
  Image      : /content/drive/MyDrive/AI-CropFieldSegmentation/Screenshot 2026-03-17 063235.png
  Checkpoint : /content/drive/MyDrive/AI-CropFieldSegmentation/result/fractal-resunet-512-v03/best.pt
  Output dir : /content/drive/MyDrive/AI-CropFieldSegmentation/output

[1/5] Loading image…


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


  Caching 0.0 GB image into RAM…
  ✓ Cached (0.00 GB)
  ✓ 1006×768  CRS=None  RAM=1.99 GB

[2/5] Running FracTAL ResUNet tiled inference…
  6 tiles  (tile=512px  overlap=128px  resolution=1.0)


FracTAL mosaic: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


  ✓ Full-image mosaic complete — extracting polygons from global mask…
    Connected components: 74 raw instances
  ✓ 34 raw detections (mosaic-first — no tile-boundary cuts)
  RAM: 1.99 GB  |  t=1.9s

[3/5] Polygon NMS…
  Polygon NMS: 34 → 34  (IoU>0.4)
  RAM: 1.99 GB  |  t=1.9s

[4/5] Georeferencing…


Georeferencing: 100%|██████████| 34/34 [00:00<00:00, 588.01it/s]

  ✓ 34 georeferenced polygons
  RAM: 1.99 GB  |  t=2.0s

[5/5] Visualization…
  Canvas: 1006×768 → 1006×768  scale=1.0000


  ✓ PNG: /content/drive/MyDrive/AI-CropFieldSegmentation/output/cropfield_fractal_segmentation.png  (1.1 MB)

[Save] Writing outputs…
  ✓ CSV     : /content/drive/MyDrive/AI-CropFieldSegmentation/output/cropfield_fractal_fields.csv
  ✓ Stats   : /content/drive/MyDrive/AI-CropFieldSegmentation/output/cropfield_fractal_statistics.json

  ✓ Done in 0.1 min  |  34 fields
  Peak RAM: 2.01 GB

